# Notebook 10: Validación en Bases de Datos

**Duración**: 40 minutos | **Nivel**: Intermedio

## Introducción

Aprende a validar datos directamente en bases de datos SQL sin cargarlos en memoria.

### Objetivos:
1. Conectar a SQLite
2. Validar tablas SQL
3. Usar expectativas SQL personalizadas
4. Optimizar validaciones en BD

In [ ]:
import great_expectations as gx
import sqlite3
import pandas as pd

# Crear BD de ejemplo
conn = sqlite3.connect("../data/temperature.db")
df = pd.read_csv("../data/temperature.csv")
df.to_sql("temperature", conn, if_exists="replace", index=False)
conn.close()

print(" Base de datos creada")

## Conectar a Base de Datos

In [ ]:
context = gx.get_context(mode="ephemeral")

# Configurar datasource SQL
datasource = context.data_sources.add_sql(
    name="sqlite_ds",
    connection_string="sqlite:///./data/temperature.db"
)

# Agregar asset (tabla)
asset = datasource.add_table_asset(name="temperature", table_name="temperature")
batch_def = asset.add_batch_definition_whole_table("batch_completo")

print(" Conectado a base de datos")

## Validar Tabla SQL

In [ ]:
suite = context.suites.add(gx.ExpectationSuite(name="validacion_sql"))

# Expectativas estándar funcionan en SQL
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="date")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="temperature",
        min_value=-50,
        max_value=50
    )
)

suite.save()

val_def = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite, name="val_sql")
)

resultado = val_def.run()
print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")

## Expectativas SQL Personalizadas

In [ ]:
# Validar con query SQL personalizada
suite_custom = context.suites.add(gx.ExpectationSuite(name="validacion_sql_custom"))

# Contar registros
suite_custom.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1,
        max_value=10000
    )
)

suite_custom.save()

val_def_custom = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_custom, name="val_custom")
)

resultado_custom = val_def_custom.run()
print(f"Validación custom: {'' if resultado_custom.success else ''}")

##  Ejercicio

Crea una validación que verifique que no hay duplicados en la columna 'date'.

In [ ]:
# TU CÓDIGO AQUÍ
pass

In [ ]:
context.build_data_docs()
context.open_data_docs()

##  Resumen

1.  GX valida datos directamente en SQL
2.  No necesitas cargar datos en memoria
3.  Expectativas estándar funcionan en SQL
4.  Puedes crear queries SQL personalizadas

